# EVT Threshold Selection and Diagnostics

This interactive notebook guides you through the process of selecting a **peaks‑over‑threshold (POT)** threshold for river discharge data **on a per‑station basis**.  Each GloFAS station in your basin may have a different distribution of discharge values, so you should repeat this analysis for **each station** listed in your basin configuration.  

Selecting an appropriate threshold is a critical step in extreme value analysis.  A low threshold may include too many moderate events and violate the asymptotic theory underlying the Generalised Pareto Distribution (GPD).  A high threshold, on the other hand, leaves too few data points to estimate the tail shape reliably.  

In this notebook you will:

* Load historical GloFAS discharge time series for your basin using the project’s configuration and data loaders.
* Visualise the time series and explore the overall distribution of discharges for the station you choose.
* Compute and plot the **mean residual life (MRL)** and **GPD parameter stability** diagnostics across a range of candidate thresholds.
* Choose an appropriate threshold and declustering window (run length) based on the diagnostics.
* Extract independent peaks above your chosen threshold for use in the GPD calibration (next notebook).

Each section contains explanatory text to help operational staff interpret the plots and make informed decisions.  Adjust the parameters in the code cells as needed to reflect your basin, calibration period, station and declustering window.  After completing the analysis for one station, repeat the steps for the next station.


In [ ]:

# Auto‑reload modules so that changes in src/ are picked up without restarting the kernel
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Ensure project source directory is on sys.path for imports
# This assumes the notebook lives in calibration/notebooks/ and src/ is two levels up
project_root = Path.cwd().resolve().parents[2]
src_path = project_root / 'src'
if src_path.as_posix() not in sys.path:
    sys.path.insert(0, src_path.as_posix())

# Import configuration loader and data loader
from philflood.domain.config import load_basin_config
from philflood.adapters.glofas import load_glofas_reanalysis_for_basin

# Import threshold analysis utilities
from philflood.models.ev.threshold_analysis import (
    extract_declust_pot,
    mean_residual_life,
    gpd_parameter_stability,
    suggest_threshold_range,
)

# Configure matplotlib for inline plots
plt.rcParams.update({
    'figure.figsize': (10, 4),
    'axes.grid': True,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
})


In [ ]:

# -----------------------------------------------------------------------------
# User inputs: edit these values to match your basin and calibration period
# -----------------------------------------------------------------------------
# Path to the basin configuration YAML file
# Example: 'ops/configs/basins/Cagayan_01.yaml'
basin_cfg_path = 'ops/configs/basins/Cagayan_01.yaml'  # TODO: update this path

# Calibration period (inclusive).  GloFAS reanalysis data are available from 1979 onwards.
start_date = '1980-01-01'
end_date = '2020-12-31'

# Declustering window in days: merge all exceedances within this window into one cluster
# Adjust this parameter if hydrological conditions dictate a shorter or longer flood response time.
run_length_days = 5  # typically between 3 and 7 days

# Range of quantiles to explore when suggesting thresholds
lower_quantile = 0.90
upper_quantile = 0.99
num_thresholds = 20


In [ ]:

# Load the basin configuration.  This dataclass defines GloFAS station IDs
# and the data_root where time series should be stored.  Ensure that
# your YAML file defines ``glofas_point_ids`` and a ``data_root`` pointing
# to the folder where extracted time series live (see the docs for details).

basin_cfg = load_basin_config(basin_cfg_path)
print(f'Loaded basin configuration for: {basin_cfg.basin_id}')
print(f'GloFAS points: {basin_cfg.glofas_point_ids}')
print(f'Data root: {basin_cfg.data_root}')

# Fetch the reanalysis discharge time series for the specified period
# The loader will attempt to read from <data_root>/glofas/timeseries/<basin_id>__discharge.parquet
# or .csv.  If no file is found it will return an empty DataFrame with NaNs.
discharge_df = load_glofas_reanalysis_for_basin(
    basin_cfg, start_date, end_date
)

# Drop completely empty columns (no data) and display an overview
empty_cols = discharge_df.columns[discharge_df.isna().all()]
if len(empty_cols) > 0:
    print(f'Warning: the following stations have no data in the requested period: {list(empty_cols)}')
    discharge_df = discharge_df.drop(columns=empty_cols)

print('Discharge data shape:', discharge_df.shape)
discharge_df.head()


In [ ]:

# -----------------------------------------------------------------------------
# Station selection
# -----------------------------------------------------------------------------
# Choose a station to analyse.  Each station may require its own threshold.
# If multiple stations are available you can set ``station_id`` to the desired
# column name (a GloFAS grid cell or station identifier).

if discharge_df.empty:
    raise RuntimeError('No discharge data available. Please check your data files or period settings.')

# Default to the first station in the DataFrame
station_id = discharge_df.columns[0]  # TODO: change to other station ID if needed
series = discharge_df[station_id]

# Plot the time series using Plotly for interactive exploration
fig_ts = px.line(
    series.reset_index(),
    x=series.index.name or 'index',
    y=station_id,
    title=f'Discharge time series at station {station_id}',
    labels={'x': 'Date', 'y': 'Discharge (m^3/s)'}
)
fig_ts.update_layout(showlegend=False, height=400)
fig_ts


In [ ]:

# Overview of the discharge distribution for the selected station.  Histograms
# are useful to see how skewed the distribution is, while the empirical
# CDF highlights the upper tail where extreme value methods will apply.

clean = series.dropna()

# Histogram
fig_hist = px.histogram(
    clean,
    nbins=50,
    title=f'Discharge histogram at station {station_id}',
    labels={'value': 'Discharge (m^3/s)', 'count': 'Frequency'},
    marginal='box'
)
fig_hist.update_layout(height=400)
fig_hist

# Empirical CDF
sorted_vals = np.sort(clean)
y_cdf = np.linspace(0, 1, len(sorted_vals), endpoint=False)
fig_cdf = go.Figure()
fig_cdf.add_trace(go.Scatter(x=sorted_vals, y=y_cdf, mode='lines', name='Empirical CDF'))
fig_cdf.update_layout(
    title=f'Empirical CDF of discharge at station {station_id}',
    xaxis_title='Discharge (m^3/s)',
    yaxis_title='Cumulative probability',
    height=400
)
fig_cdf


In [ ]:

# Compute a range of candidate thresholds based on quantiles
thresholds = suggest_threshold_range(
    series,
    lower_quantile=lower_quantile,
    upper_quantile=upper_quantile,
    num=num_thresholds
)
print('Candidate thresholds:')
print(thresholds)

# Mean residual life
mrl_df = mean_residual_life(series, thresholds, run_length_days=run_length_days)

# Parameter stability
stability_df = gpd_parameter_stability(series, thresholds, run_length_days=run_length_days)

# Merge for plotting
diag_df = pd.merge(mrl_df, stability_df, on='threshold', how='outer')
diag_df


In [ ]:

# Combined diagnostic plots using Plotly
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=(
        'Mean Residual Life (MRL)',
        'GPD Shape Parameter Stability'
    )
)

# MRL plot
fig.add_trace(
    go.Scatter(
        x=diag_df['threshold'],
        y=diag_df['mean_excess'],
        mode='lines+markers',
        name='Mean Excess',
        marker=dict(color='darkblue')
    ),
    row=1, col=1
)

# Number of exceedances (secondary axis) -- using suffix _x because of merge columns names
fig.add_trace(
    go.Scatter(
        x=diag_df['threshold'],
        y=diag_df['n_exceedances_x'],
        mode='lines+markers',
        name='Number of exceedances',
        marker=dict(color='orange'),
        yaxis='y2'
    ),
    row=1, col=1
)

# Shape parameter stability
fig.add_trace(
    go.Scatter(
        x=diag_df['threshold'],
        y=diag_df['shape'],
        mode='lines+markers',
        name='Shape parameter (xi)',
        marker=dict(color='darkgreen')
    ),
    row=2, col=1
)

# Layout adjustments
fig.update_layout(
    height=600,
    title_text=f'Threshold diagnostics for station {station_id}',
    xaxis=dict(title='Threshold (discharge units)'),
    yaxis=dict(title='Mean excess'),
    yaxis2=dict(title='Number of exceedances', overlaying='y', side='right'),
    xaxis2=dict(title='Threshold (discharge units)'),
    yaxis3=dict(title='Shape parameter (xi)'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)
fig


In [ ]:

# -----------------------------------------------------------------------------
# Select a threshold and extract independent peaks
# -----------------------------------------------------------------------------
# Based on the diagnostic plots above, choose a threshold that falls in a
# region where the MRL is approximately linear and the shape parameter is
# relatively stable.  Update the value of `selected_threshold` below.
selected_threshold = float(thresholds[int(len(thresholds) * 0.5)])  # Example: median candidate
print(f'Selected threshold for station {station_id}: {selected_threshold}')

# Extract independent peaks using the selected threshold and run_length
peaks = extract_declust_pot(series, selected_threshold, run_length_days=run_length_days)
print(f'Number of independent peaks at station {station_id}: {len(peaks)}')

# Display the first few peaks
peaks.head()


In [ ]:

# Overlay the selected peaks on the original time series to visually confirm
# the declustering.  Use this plot to ensure that clusters are separated by
# at least run_length_days days.

fig_overlay = go.Figure()
fig_overlay.add_trace(
    go.Scatter(
        x=series.index,
        y=series,
        mode='lines',
        name='Discharge time series',
        line=dict(color='lightgrey')
    )
)
fig_overlay.add_trace(
    go.Scatter(
        x=peaks.index,
        y=peaks,
        mode='markers',
        name='Independent peaks',
        marker=dict(color='red', size=6, symbol='circle')
    )
)
fig_overlay.update_layout(
    title=f'Independent peaks above threshold {selected_threshold} at station {station_id}',
    xaxis_title='Date',
    yaxis_title='Discharge (m^3/s)',
    height=400
)
fig_overlay


## Next steps: fitting the Generalised Pareto Distribution

Once you have selected a threshold and extracted the independent peaks for **each station**, the next step
is to fit a Generalised Pareto Distribution (GPD) to the exceedances.  This is
covered in the next calibration notebook `02_evt_threshold_basin_X_gpd_fit.ipynb`.

**Remember** to record your chosen thresholds and run length in the basin
configuration YAML under the `evt:` section for each station.  If you
choose different thresholds for different stations, you may need to
extend the configuration structure to store a mapping from station to
threshold.  Alternatively, select a representative station for your basin
and document your choice in the project notes.

You may also wish to export the peaks series to a file for later use.  For
example:

```python
peaks.to_csv(f'data/processed/ev/{basin_cfg.basin_id}/peaks_{station_id}.csv')
```

This notebook is intended as a guide; feel free to adapt it to your basin
specifics, explore different run lengths, and refine your threshold choice.  After
completing the analysis for one station, rerun the notebook or select a new
station in cell 5 and repeat the diagnostics.
